In [ ]:
from sklearn.datasets import fetch_openml
import matplotlib as mpl
from matplotlib import pyplot as plt
import numpy as np
import torch
from sklearn.base import BaseEstimator
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import SGDClassifier

mnist = fetch_openml("mnist_784", version=1)
mnist.keys()

In [ ]:
X, y = mnist["data"], mnist["target"]
print(torch.cuda.is_available())

In [ ]:
if (type(X) != "numpy.ndarray"):
    X = X.to_numpy()
print(type(X))
some_digit = X[0]
some_digit_image = some_digit.reshape(28, 28)

plt.imshow(some_digit_image, cmap=mpl.cm.binary, interpolation="nearest")
plt.axis("off")
plt.show()

In [ ]:
y[0]

In [ ]:
y = y.astype(np.uint8)

In [ ]:
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

In [ ]:
y_train_5 = y_train == 5
y_test_5 = y_test == 5
print(np.unique(y_test_5))

In [ ]:
sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_5)

In [ ]:
sgd_clf.predict([some_digit])

### Implementing Cross-Validation

In [ ]:
skfolds = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)
for train_idx, test_idx in skfolds.split(X_train, y_train_5):
    clone_clf = clone(sgd_clf)
    X_train_folds = X_train[train_idx]
    y_train_folds = y_train_5[train_idx]
    X_test_folds = X_train[test_idx]
    y_test_folds = y_train_5[test_idx]
    clone_clf.fit(X_train_folds, y_train_folds)
    y_pred = clone_clf.predict(X_test_folds)
    n_correct = sum(y_pred == y_test_folds)
    print(n_correct / len(y_pred))

In [ ]:
scores = cross_val_score(sgd_clf, X_train, y_train_5, cv=3)

print(scores)
print(scores.mean())

In [ ]:
class Never5Classifier(BaseEstimator):
    def fit(self, X, y=None):
        pass

    def predict(self, X):
        return np.zeros((len(X), 1), dtype=bool)


never_5_clf = Never5Classifier()
y_train_pred = cross_val_score(never_5_clf, X_train, y_train_5, cv=3, scoring="accuracy")

### Implementing Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict

y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)


confusion_matrix(y_train_5, y_train_pred)

In [ ]:
print(y_train_5.shape)
print(y_train_pred.shape)

In [ ]:
y_train_perfect_prediction = y_train_5
confusion_matrix(y_train_5, y_train_perfect_prediction)

In [ ]:
from sklearn.metrics import precision_score,recall_score
precision = precision_score(y_train_5,y_train_pred)

In [ ]:
recall = recall_score(y_train_5,y_train_pred)

In [ ]:
f1 = (2*precision*recall)/(precision+recall)
print(f1)

In [ ]:
from sklearn.metrics import f1_score

f1_score(y_train_5,y_train_pred)

In [ ]:
y_scores = sgd_clf.decision_function([some_digit])
y_scores

In [ ]:
threshold = 8000
y_some_digit_pred = y_scores > threshold
y_some_digit_pred

In [ ]:
y_scores = cross_val_predict(
    sgd_clf, X_train, y_train_5, cv=3, method="decision_function"
)

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, threshold = precision_recall_curve(y_train_5, y_scores)


def plot_precision_recall_vs_threshold(precisions, recalls, thresholds):
    plt.plot(thresholds, precisions[:-1], "b--", label="Precision")
    plt.plot(thresholds, recalls[:-1], "g-", label="Recall")
    [...]  # highlight the threshold, add the legend, axis label and grid


plot_precision_recall_vs_threshold(precision, recall, threshold)

plt.legend()
plt.show()

In [ ]:
plt.plot(precision, label="precision")
plt.plot(recall,label="recall")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve
fpr,tpr,thresholds = roc_curve(y_train_5,y_scores)
def plot_roc_curve(fpr, tpr, label=None):
    plt.plot(fpr, tpr, linewidth=2, label=label)
    plt.plot([0, 1], [0, 1], 'k--') # dashed diagonal
    [...] # Add axis labels and grid
plot_roc_curve(fpr,tpr,thresholds)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_clf = RandomForestClassifier(random_state=42)
y_probas_forest = cross_val_predict(
    forest_clf, X_train, y_train_5, cv=3, method="predict_proba"
)
y_scores_forest = y_probas_forest[:, 1]  # score = proba of positive class
fpr_forest, tpr_forest, thresholds_forest = roc_curve(y_train_5, y_scores_forest)

plt.plot(fpr, tpr, "b:", label="SGD")
plot_roc_curve(fpr_forest, tpr_forest, "Random Forest")
plt.legend(loc="lower right")
plt.show()

In [ ]:
some_digit_scores = sgd_clf.decision_function([some_digit])